# Full problem walkthrough

This notebook creates a very simple 2-group anisotropicscattering problem to show how to construct operators and take derivatives.
In principle, this shows the entire working code in one notebook!

Problem is described also in [Bogaarts et al. (2025)](https://iopscience.iop.org/article/10.1088/1741-4326/adddc9/pdf). 

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
jax.config.update("jax_platforms", "cpu")
jax.config.update("jax_enable_x64", True)
import jax_sn
import equinox as eqx
from typing import List

We are first going to create the materials:


In [ ]:
from jax_sn.domain.cross_sections import CrossSectionData, CrossSectionsIndexed

water = CrossSectionData(
    total = jnp.array([15.2090, 24.4140]),
    scattering = jnp.array([
        [
            [5.5069, 0.0],
            [6.6227, 11.4080]
        ],
        [
            [1.158567, 0.0],
            [1.1234767, 2.4733]
        ]
    ])
)
iron = CrossSectionData(
    total = jnp.array([30.3110, 26.8760]),
    scattering = jnp.array([
        [
            [18.2500, 0.0],
            [6.45294, 21.3570]
        ],
        [
            [4.36, 0.0],
            [-0.003508, 2.3457]
        ]        
    ])
)
void = CrossSectionData(
    total = jnp.array([1e-10, 1e-10]),
    scattering = jnp.zeros((2, 2, 2))
)


We now create the domain from the following regions:

```text
0   - 0.2 m: Water
0.2 - 0.3 m: Water with source
0.3 - 0.8 m: Water
0.8 - 1.0 m: Iron
1.0 - 1.1 m: Void
1.1 - 1.2 m: Water
1.2 - 1.3 m: Void
```

We just use a simple discretization with a set number of elements per 10 cm and build from there (we hide the 
details of creating the mesh and material indices)

In [ ]:
elements_per_10_cm = 2
base_region_boundaries  = jnp.array([0.0, 0.2, 0.3, 0.8, 1.0, 1.1, 1.2, 1.3])
base_region_sizes       = [a - b for a,b in zip(base_region_boundaries[1:], base_region_boundaries[:-1])]
base_elements_per_region =  [int(round( size * 10 * elements_per_10_cm)) for size in base_region_sizes]



In [ ]:
@eqx.filter_jit
def create_vertices(region_boundaries : jnp.array, elements_per_region : List[int]):
    region_sizes          = [a - b for a,b in zip(region_boundaries[1:], region_boundaries[:-1])]
    

    vertices = jnp.concatenate([
        jnp.linspace(start, start + size, n_elements, endpoint=False) 
        for start, size, n_elements in zip(region_boundaries[:-1], region_sizes, elements_per_region)
    ] + [jnp.array([region_boundaries[-1]])])[:, None]
    return vertices

@eqx.filter_jit
def create_anisotropic_scattering_domain(region_boundaries : jnp.array, elements_per_region : List[int]):
    from jax_sn.domain import Domain

    
    vertices = create_vertices(region_boundaries, elements_per_region)
    
    connectivity = jnp.array([[i, i+1] for i in range(sum(elements_per_region))])

    water_idx = 0
    iron_idx  = 1
    void_idx  = 2

    material_indices = [water_idx, water_idx, water_idx, iron_idx, void_idx, water_idx, void_idx]

    material_indices_elements =jnp.concatenate([
        jnp.full(n_elements, material) for n_elements, material in zip(elements_per_region, material_indices)
    ])
    source_regions = [0, 1, 0, 0, 0, 0, 0]

    source_base = jnp.concatenate([
        jnp.full(n_elements, source) for n_elements, source in zip(elements_per_region, source_regions)
    ])
    
    source_values = jnp.array([9.7702e4, 4.5451e5])

    source = source_base[None, :] * source_values[:, None] 
    return (vertices, connectivity), material_indices_elements, source            

In [ ]:

mesh, material_indices, source = create_anisotropic_scattering_domain(base_region_boundaries,
                                     base_elements_per_region)
from jax_sn.domain import Domain

cross_sections_indexed = CrossSectionsIndexed(material_indices, 
                                             CrossSectionData.from_list([water, iron, void]))

domain  =  Domain.from_mesh_and_cross_sections(mesh = mesh, 
                                               face_template = jnp.array([[0],[1]]),
                                               cross_sections_indexed = cross_sections_indexed)

The face template determines which face contains which vertices (must be consistent with your Element!)

Now we construct a SolutionDomain from it:

In [ ]:
from jax_sn.solution_domain import BasixLagrangianSimplex, SolutionDomain

element_order   = 1
element         = BasixLagrangianSimplex(element_order, 1)
solution_domain = SolutionDomain.from_element_and_domain(element = element, domain = domain)


Now we are going to construct the multigroup operator! As mentioned, this requires a particular strategy. 
We will use a simple (non-optimal) strategy to explain it, where we decompose the multigroup 
operator into single-group operators, which are decomposed into their own respective operators.

In [ ]:
from jax_sn.operator_creation.default_strategies import *

mg_strategy = MultiGroupChainGatherScanStrategy(
    single_group_chain_strategy = ComposedSingleGroupChainStrategy(
        transport_strategy           = VmapAnglesForward(inner = VmapForward()),
        transport_inversion_strategy = VmapAnglesInversion(inner = VmapSweep()),
        moment_strategy              = EinsumFluxMoment(),
        discrete_strategy            = EinsumFluxMoment(),
        mass_strategy                = LaxMass(),        
    ),
    single_group_gather_scatter_strategy = BroadcastedSingleGroupGather(),
    checkpoint = True
)


What does this mean? 

### Multi group strategy

The multigroup version of the operator is:

$$ \mathbf{I} - \mathbf{M} \mathbf{T}^{-1} \mathbf{H} \mathbf{D}\mathbf{S} $$

{py:class}`~jax_sn.operators.multi_group_operator.MultiGroupChainGatherScanStrategy` means instead of computing one big operator, we decompose it into two single group operators and combine the results using `jax.scan`. In other words:

$$ \text{for all } g: \quad \mathbf{I} - \underbrace{\mathbf{M}_g \mathbf{T}^{-1}_g \mathbf{H}_g \mathbf{D}_g}_{\text{chain}}\underbrace{\mathbf{S}_g}_{\text{gather-scatter}} $$

Both the chain and the gather-scatter are single-group operators, which have their own strategies.

#### Single group scatter strategy

This operator is implemented through a simple broadcasting gathering operator.

#### Single group chain strategy

The chain operator 

$$ \mathbf{M}_g \mathbf{T}^{-1}_g \mathbf{H}_g \mathbf{D}_g$$

is a {py:class}`~jax_sn.operators.single_group_operator.ComposedSingleGroupChainStrategy`, which means each operator is defined separately and the results are combined (instead of a single strategy that defines all operators at the same time)

##### Moment and discrete strategy
These operators are essentially just a batched matrix multiplication, so they share the same type of strategies: both 
einsums ({py:class}`~jax_sn.operators.moment_operator.EinsumFluxMoment`).

##### Mass strategy
The mass operator is also a batched matrix operator, but operators on different dimensions of the array. Here it is implemented as a Lax matrix product ({py:class}`~jax_sn.operators.mass_operator.LaxMass`).

#### Transport operator

The transport operator $\mathbf{T}^{-1}_g$ is somewhat special: it is an inversion in its own right and it also maps over all angles! Differentiability requires a forward strategy as well: here, we just vmap over the angles ({py:class}`~jax_sn.operators.transport_operator.VmapAnglesForward`) and vmap over all elements ({py:class}`~jax_sn.operators.transport_operator.VmapForward`), while the inversion strategy also vmaps over the angles ({py:class}`~jax_sn.operators.transport_operator.VmapAnglesInversion`) using as single-angle function {py:class}`~jax_sn.operators.transport_operator.VmapSweep`.

Finally, `checkpoint = True`  means that we checkpoint the multigroup operator for each group. This reduces memory cost significantly for differentiated solves. See [here](https://docs.jax.dev/en/latest/gradient-checkpointing.html) for more information


We provide some useful defaults for the multi group strategy in `jax_sn.operator_creation.default_strategies` (of which this is one):

In [ ]:
DefaultMultiGroupStrategy.default_multi_group_strategy()

In [ ]:
DefaultMultiGroupStrategy.default_multi_group_strategy(batch_size = 1024)

We now create the multigroup operator. This function computes all the required setup for it from the solution domain (including topological sort and FEM matrices) and returns 
a {py:class}`~jax_sn.operators.multi_group_operator.MultiGroupOperator` object (which as before, contains data and strategy)

In [ ]:
from jax_sn.operators.multi_group_operator import MultiGroupOperator

from jax_sn.operator_creation import create_multi_group_operator

quadrature_order = 3 
quadrature_set   = jax_sn.quadrature_set.create_tn_quadrature_set(quadrature_order)

mgop = create_multi_group_operator(solution_domain = solution_domain,
                                   quadrature_set   = quadrature_set,
                                   multi_group_strategy = mg_strategy)

print(mgop)

Before solving we create an isotropic source (we repeat along the basis to have a flat source within each element):

In [ ]:
from jax_sn.solvers.right_hand_side import TotalSource

isotropic_src = jnp.repeat(source[..., None], mgop.data.n_basis, axis=-1)

total_source = TotalSource.from_operator_and_isotropic_source(mgop, isotropic_src)


## Solving

Solving is easy in this formulation! We first create the correct right-hand-side:

$$ \mathbf{b} = \mathbf{M} \mathbf{T}^{-1} \mathbf{D} \mathbf{H} [q_{g\ell ei}] $$

This is precisely the chain strategy in multigroup form. We could create a separate multigroup version of it, but here we just scan the single-group chain strategy we already made.


In [ ]:
from jax_sn.operators.multi_group_operator.composed_multi_group_strategies import MultiGroupChainScanStrategy
mg_chain = MultiGroupChainScanStrategy(mgop.strategy.single_group_chain_strategy)

We will create a top-level function to solve, including the right-hand-side operator, since this allows for a 
consistent derivative later (note: this function actually exists in the package, but here we construct it to show how it works).

In [ ]:
import lineax as lx
import equinox as eqx

@eqx.filter_jit
def solve_anisotropic(mgop : MultiGroupOperator, total_source : TotalSource, solver : lx.AbstractLinearSolver, mg_chain  : MultiGroupChainScanStrategy):        
    rhs   = mg_chain.forward(mgop.data, total_source)
    return lx.linear_solve(operator = mgop, vector = rhs, solver = solver)


In [ ]:
solution = solve_anisotropic(mgop, total_source, lx.BiCGStab(rtol = 1e-10, atol = 1e-10, max_steps = 20), mg_chain)
print("Solution converged in", solution.stats['num_steps'], "steps")

We can easily plot it using the built-in interpolation:

In [ ]:
def interpolate_and_plot(solution_domain, solution, linestyle = '-'):
    x           = jnp.linspace(0,1.3,200)[:, None] # make it batch of 200 points in 1D
    scalar_flux = solution_domain.interpolate(x, solution.value[:,0]) # interpolate *scalar flux l=0*
    

    plt.semilogy(x[:,0], scalar_flux.T[:,0], label= ["$E_0$"], color="C0", linestyle=linestyle)
    plt.semilogy(x[:,0], scalar_flux.T[:,1], label= ["$E_1$"], color="C1", linestyle=linestyle)
    plt.legend()
    plt.xlabel("x")
    _ = plt.ylabel(r"$\phi$")
    plt.xlim(0,1.3)

interpolate_and_plot(solution_domain, solution)

Clearly, the linear elements are not so smooth in this regime!
Let's do third order elements, re-using the same strategies.

In [ ]:
element_order_p3   = 3
element_p3         = BasixLagrangianSimplex(element_order_p3, 1)
solution_domain_p3 = SolutionDomain.from_element_and_domain(element = element_p3, domain = domain)

mgop_p3 = create_multi_group_operator(solution_domain = solution_domain_p3,
                                   quadrature_set   = quadrature_set,
                                    multi_group_strategy = mg_strategy)

total_source_p3 = TotalSource.from_operator_and_isotropic_source(mgop_p3, 
                                                              total_isotropic_source = jnp.repeat(source[..., None], mgop_p3.data.n_basis, axis=-1))

solution_p3 = solve_anisotropic(mgop_p3, total_source_p3, lx.BiCGStab(rtol = 1e-10, atol = 1e-10, max_steps = 20), mg_chain)
print("Solution converged in", solution_p3.stats['num_steps'], "steps")

interpolate_and_plot(solution_domain_p3, solution_p3)

interpolate_and_plot(solution_domain, solution, linestyle = '--')


The 3th order elements are much smoother!

## Obtaining Derivatives

Derivatives requires also parametrising the Jacobian calculation of each element. 
We show how to do it here by parametrizing the region sizes, calculating the new vertices, 
and updating the multigroup operator with the new vertices, re-calculating the relevant precomputed geometric data.

In this 1D case, the topological sorting is the same, but in 3D or 2D, it can be recalculated as well.

In [ ]:
from jax_sn.parametrisation.multigroup_parametrisation import reconstruct_multigroup_operator_from_vertices
import equinox as eqx

@eqx.filter_jit
def solve_and_sum(new_region_boundaries, mgop, total_source, mg_chain):
    vertices = create_vertices(new_region_boundaries, base_elements_per_region)    
    mgop_param = reconstruct_multigroup_operator_from_vertices(mgop, vertices, replace_topological_sort = False) # no toposort change in linear domain!
    solution = solve_anisotropic(mgop_param, total_source, lx.BiCGStab(rtol = 1e-10, atol = 1e-10, max_steps = 40), mg_chain)
    return jnp.sum(solution.value)

grad_solve_and_sum = eqx.filter_jit(eqx.filter_grad(solve_and_sum))

We can check the results:

In [ ]:
print("Result", base_result := solve_and_sum(base_region_boundaries, mgop, total_source, mg_chain))
print("Gradient", grad_solve_and_sum(base_region_boundaries, mgop, total_source, mg_chain))
print("Finite difference", (solve_and_sum(base_region_boundaries.at[0].add(1e-6), mgop, total_source, mg_chain) - base_result)/ 1e-6)

As a final sanity check, we can check whether the above really is the same as constructing it from scratch:


In [ ]:
modified_boundaries = [0.0, 0.1, 0.3, 0.8, 1.0, 1.1, 1.25, 1.3]
mesh_m, material_m, source_m = create_anisotropic_scattering_domain(modified_boundaries, base_elements_per_region)

domain_m = Domain.from_mesh_and_cross_sections(mesh_m, domain.face_template, domain.cross_sections_indexed)



solution_domain_m = SolutionDomain.from_element_and_domain(element = element, domain = domain_m)

mgop_m =  create_multi_group_operator(solution_domain = solution_domain_m,
                            quadrature_set   = quadrature_set,
                            multi_group_strategy = mg_strategy)

total_source_m = TotalSource.from_operator_and_isotropic_source(mgop_m, 
                                                              total_isotropic_source = jnp.repeat(source_m[..., None], mgop_m.data.n_basis, axis=-1))

solution_m = solve_anisotropic(mgop_m   , total_source_m, lx.BiCGStab(rtol = 1e-10, atol = 1e-10, max_steps = 20), mg_chain)

interpolate_and_plot(solution_domain_m, solution_m)

print("From scratch:      ", solve_and_sum(modified_boundaries, mgop_m, total_source_m, mg_chain))
print("Modified from base:", solve_and_sum(modified_boundaries, mgop, total_source, mg_chain))

Similarly, the gradients are equal:

In [ ]:
print("From scratch:      ", grad_solve_and_sum(jnp.array(modified_boundaries), mgop_m, total_source_m, mg_chain))
print("Modified from base:", grad_solve_and_sum(jnp.array(modified_boundaries), mgop, total_source, mg_chain))

Finally, let us also check that the adjoint method really just solves back and forth by timing it:

In [ ]:
%timeit solve_and_sum(base_region_boundaries, mgop, total_source, mg_chain).block_until_ready()

In [ ]:
%timeit grad_solve_and_sum(base_region_boundaries, mgop, total_source, mg_chain).block_until_ready()